### Invoice Data

This stage uploads pre-generated supplier invoice PDFs to a Unity Catalog volume
and loads the structured metadata as Delta tables. Each PDF is a real supplier invoice
with line items, payment terms, and financial flags (discrepancies, late fees, SLA penalties).

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

In [ ]:
# Drop any stale managed tables that conflict with DLT gold table names.
# These may exist from earlier runs before the pipeline was introduced.
PIPELINE_GOLD_TABLES = ["invoices", "line_items", "spend_by_supplier", "spend_by_category",
                        "payment_aging", "invoice_exceptions", "supplier_scorecard", "discount_analysis",
                        "contracts"]
for tbl in PIPELINE_GOLD_TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.procurement.{tbl}")
    print(f"  Dropped (if existed): {CATALOG}.procurement.{tbl}")
print("✅ Cleared stale managed tables")

##### Create catalog, schema, and volumes for procurement documents

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.procurement")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.procurement.invoices")
print(f"✅ Created schema {CATALOG}.procurement and volume invoices")

##### Copy invoice PDFs to the Unity Catalog volume

In [ ]:
import os
import glob

pdf_source_dir = os.path.abspath("../data/invoices/pdfs")
volume_path = f"/Volumes/{CATALOG}/procurement/invoices"

pdf_files = glob.glob(os.path.join(pdf_source_dir, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files to upload")

for pdf_file in pdf_files:
    filename = os.path.basename(pdf_file)
    with open(pdf_file, "rb") as src:
        with open(f"{volume_path}/{filename}", "wb") as dst:
            dst.write(src.read())
    print(f"  Uploaded: {filename}")

print(f"✅ Uploaded {len(pdf_files)} invoice PDFs to {volume_path}")

##### Load invoice metadata into Delta tables

In [ ]:
import json
from datetime import date as dt_date
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, BooleanType, ArrayType, DateType
)

metadata_path = os.path.abspath("../data/invoices/invoice_metadata.json")
with open(metadata_path) as f:
    metadata = json.load(f)

print(f"Loaded {len(metadata['invoices'])} invoices from {len(metadata['suppliers'])} suppliers")

In [ ]:
# --- Suppliers table ---
supplier_rows = []
for s in metadata["suppliers"]:
    supplier_rows.append({
        "supplier_id": s["supplier_id"],
        "name": s["name"],
        "address": s["address"],
        "city_state_zip": s["city_state_zip"],
        "phone": s["phone"],
        "email": s["email"],
        "tax_id": s["tax_id"],
        "category": s["category"],
        "payment_terms": s["payment_terms"],
        "contract_id": s["contract_id"],
    })

supplier_schema = StructType([
    StructField("supplier_id", StringType()),
    StructField("name", StringType()),
    StructField("address", StringType()),
    StructField("city_state_zip", StringType()),
    StructField("phone", StringType()),
    StructField("email", StringType()),
    StructField("tax_id", StringType()),
    StructField("category", StringType()),
    StructField("payment_terms", StringType()),
    StructField("contract_id", StringType()),
])

df_suppliers = spark.createDataFrame(supplier_rows, schema=supplier_schema)
df_suppliers.write.mode("overwrite").saveAsTable(f"{CATALOG}.procurement.suppliers")
print(f"✅ Created suppliers table with {df_suppliers.count()} rows")

In [ ]:
# --- Invoices table (one row per invoice, totals pre-computed) ---
invoice_rows = []
for inv in metadata["invoices"]:
    invoice_rows.append({
        "invoice_id": inv["invoice_id"],
        "supplier_id": inv["supplier_id"],
        "invoice_date": dt_date.fromisoformat(inv["invoice_date"]),
        "due_date": dt_date.fromisoformat(inv["due_date"]),
        "early_pay_deadline": dt_date.fromisoformat(inv["early_pay_deadline"]) if inv.get("early_pay_deadline") else None,
        "payment_date": dt_date.fromisoformat(inv["payment_date"]) if inv.get("payment_date") else None,
        "purchase_order": inv["purchase_order"],
        "delivery_date": dt_date.fromisoformat(inv["delivery_date"]) if inv.get("delivery_date") else None,
        "status": inv["status"],
        "flags": inv.get("flags", []),
        "discount_pct": float(inv.get("discount_pct", 0.0)),
        "discount_applied": bool(inv.get("discount_applied", False)),
        "volume_discount_pct": float(inv.get("volume_discount_pct", 0.0)),
        "sla_penalty_pct": float(inv.get("sla_penalty_pct", 0.0)),
        "late_fee_days_overdue": int(inv.get("late_fee_days_overdue", 0)),
        "subtotal": float(inv["subtotal"]),
        "contract_subtotal": float(inv["contract_subtotal"]),
        "discount_amount": float(inv["discount_amount"]),
        "missed_discount": float(inv["missed_discount"]),
        "volume_discount_owed": float(inv["volume_discount_owed"]),
        "sla_penalty": float(inv["sla_penalty"]),
        "late_fee": float(inv["late_fee"]),
        "price_discrepancy": float(inv["price_discrepancy"]),
        "total_due": float(inv["total_due"]),
        "pdf_filename": inv["pdf_filename"],
    })

invoice_schema = StructType([
    StructField("invoice_id", StringType()),
    StructField("supplier_id", StringType()),
    StructField("invoice_date", DateType()),
    StructField("due_date", DateType()),
    StructField("early_pay_deadline", DateType()),
    StructField("payment_date", DateType()),
    StructField("purchase_order", StringType()),
    StructField("delivery_date", DateType()),
    StructField("status", StringType()),
    StructField("flags", ArrayType(StringType())),
    StructField("discount_pct", DoubleType()),
    StructField("discount_applied", BooleanType()),
    StructField("volume_discount_pct", DoubleType()),
    StructField("sla_penalty_pct", DoubleType()),
    StructField("late_fee_days_overdue", IntegerType()),
    StructField("subtotal", DoubleType()),
    StructField("contract_subtotal", DoubleType()),
    StructField("discount_amount", DoubleType()),
    StructField("missed_discount", DoubleType()),
    StructField("volume_discount_owed", DoubleType()),
    StructField("sla_penalty", DoubleType()),
    StructField("late_fee", DoubleType()),
    StructField("price_discrepancy", DoubleType()),
    StructField("total_due", DoubleType()),
    StructField("pdf_filename", StringType()),
])

df_invoices = spark.createDataFrame(invoice_rows, schema=invoice_schema)
df_invoices.write.mode("overwrite").saveAsTable(f"{CATALOG}.procurement.invoices_raw")
print(f"✅ Created invoices_raw table with {df_invoices.count()} rows")

In [ ]:
# --- Invoice line items table (one row per line item) ---
line_item_rows = []
for inv in metadata["invoices"]:
    for item in inv["line_items"]:
        line_item_rows.append({
            "invoice_id": inv["invoice_id"],
            "supplier_id": inv["supplier_id"],
            "description": item["description"],
            "qty": float(item["qty"]),
            "unit": item["unit"],
            "unit_price": float(item["unit_price"]),
            "contract_price": float(item["contract_price"]),
        })

line_item_schema = StructType([
    StructField("invoice_id", StringType()),
    StructField("supplier_id", StringType()),
    StructField("description", StringType()),
    StructField("qty", DoubleType()),
    StructField("unit", StringType()),
    StructField("unit_price", DoubleType()),
    StructField("contract_price", DoubleType()),
])

df_line_items = spark.createDataFrame(line_item_rows, schema=line_item_schema)
df_line_items.write.mode("overwrite").saveAsTable(f"{CATALOG}.procurement.invoice_line_items")
print(f"✅ Created invoice_line_items table with {df_line_items.count()} rows")

##### Register resources with uc_state for cleanup

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

print("✅ Invoice data stage complete")